<a href="https://colab.research.google.com/github/RaniaChaudhry511/AI-ML-learning-assistant/blob/main/Transformer_GPT_Chatbot_System_Prompts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import gradio as gr

from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# ==========================================
# 1. MODEL
# ==========================================

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

In [ ]:
# ==========================================
# 2. TOKENIZER
# ==========================================

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

Loading tokenizer...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
# ==========================================
# 3. TRANSFORMER MODEL
# ==========================================

print("Loading Transformer model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")

Loading Transformer model...


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
# ==========================================
# 4. CHATBOT FUNCTION
# ==========================================

def chatbot(message, history):

    # ------------------------------
    # System instructions
    # ------------------------------

    messages = [
        {
            "role": "system",
            "content": """
You are a helpful general-purpose AI assistant.

Answer financial questions clearly and accurately.
Explain difficult financial concepts in simple language.
You can help with personal finance, budgeting, saving,
investing, stocks, bonds, banking, loans, financial markets,
economics, and general financial knowledge.
Always explain risks when discussing financial decisions.

"""
        }
    ]


    # ------------------------------
    # Add previous conversation
    # ------------------------------

    for chat in history:

        if chat["role"] in ["user", "assistant"]:

            messages.append({
                "role": chat["role"],
                "content": chat["content"]
            })


    # ------------------------------
    # Add current user message
    # ------------------------------

    messages.append({
        "role": "user",
        "content": message
    })


    # ==========================================
    # PHASE 1
    # CHAT TEMPLATE / TOKENIZATION PREPARATION
    # ==========================================

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


    # ==========================================
    # PHASE 2
    # TEXT → TOKEN IDs
    # ==========================================

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)


    # ==========================================
    # PHASE 3 + PHASE 4
    # TRANSFORMER + NEXT TOKEN PREDICTION
    # ==========================================

    with torch.no_grad():

        output_ids = model.generate(

            **inputs,

            max_new_tokens=300,

            temperature=0.7,

            top_p=0.9,

            do_sample=True,

            repetition_penalty=1.1
        )


    # ==========================================
    # GET ONLY NEW TOKENS
    # ==========================================

    new_tokens = output_ids[
        0,
        inputs["input_ids"].shape[1]:
    ]


    # ==========================================
    # PHASE 5
    # TOKEN IDs → HUMAN TEXT
    # ==========================================

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    )

    return response

In [ ]:
# ==========================================
# 5. GRADIO USER INTERFACE
# ==========================================

demo = gr.ChatInterface(

    fn=chatbot,

    title="💰 Finance AI Assistant",

    description="""
    A finance-focused AI assistant built using
    Hugging Face Transformers and Gradio.
    Ask questions about personal finance, budgeting,
    saving, investing, stocks, banking, loans, and
    financial markets.
    """,

    examples=[
        "What is a budget and how can I create one?",
        "Explain stocks and bonds in simple language.",
        "What is the difference between saving and investing?",
        "What are the risks of investing in stocks?",
        "Explain how a bank loan works."
    ],

    save_history=True
)

In [ ]:
# ==========================================
# 6. START APPLICATION
# ==========================================

demo.launch()

In [ ]:
# Launch Gradio app
demo.launch(share=True, debug=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://323e7b39ad3c45c976.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
